In [1]:
import torch
from pathlib import Path

In [2]:
# Get checkpoints files from the range 
ckpt_dir = Path("/data/ratna_retrain/eval") # RSG - changed from /data/OM_checkpoints
steps = list(range(87500, 137501, 12500))  # RSG - change this for a new range
ckpt_paths = [ ckpt_dir / f"training_{step}" / "teacher_checkpoint.pth" for step in steps]
print(f"Checkpoints to average: {steps}")
print(f"Number of paths: {len(ckpt_paths)}")

Checkpoints to average: [87500, 100000, 112500, 125000, 137500]
Number of paths: 5


In [3]:
# Filter and load checkpoints
state_dicts = [torch.load(str(p), map_location='cpu') for p in ckpt_paths if p.exists()]
print(f"Loaded {len(state_dicts)} checkpoints")


Loaded 5 checkpoints


In [4]:
teacher_dicts = [sd["teacher"] for sd in state_dicts]

In [5]:
print(len(teacher_dicts), teacher_dicts[0])

5 OrderedDict([('backbone.cls_token', tensor([[[-0.0156,  0.0013,  0.0059,  ...,  0.0012, -0.0134,  0.0014]]])), ('backbone.pos_embed', tensor([[[-0.0156,  0.0013,  0.0059,  ...,  0.0012, -0.0134,  0.0014],
         [ 0.0307,  0.0023, -0.0289,  ..., -0.0044, -0.0230,  0.0045],
         [ 0.0025, -0.0018,  0.0008,  ..., -0.0013,  0.0017,  0.0011],
         ...,
         [-0.0143,  0.0071, -0.0270,  ..., -0.0015,  0.0074, -0.0028],
         [ 0.0023,  0.0058, -0.0032,  ...,  0.0040,  0.0021,  0.0006],
         [-0.0011, -0.0003, -0.0118,  ...,  0.0020,  0.0046,  0.0026]]])), ('backbone.register_tokens', tensor([[[ 0.0063,  0.0024,  0.0085,  ...,  0.0006, -0.0242,  0.0024],
         [-0.1895,  0.0021, -0.0045,  ..., -0.0005,  0.0378,  0.0035],
         [-0.0175, -0.0023, -0.0736,  ..., -0.0065,  0.0409,  0.0041],
         [ 0.0204,  0.0035,  0.0025,  ..., -0.0017, -0.0711,  0.0021]]])), ('backbone.mask_token', tensor([[-0.1287,  0.0035,  0.0028,  ..., -0.0005, -0.0261,  0.0049]])), ('back

In [6]:
averaged_state_dict = {}
for key in teacher_dicts[0].keys():
    averaged_state_dict[key] = sum(td[key] for td in teacher_dicts) / len(teacher_dicts)

In [7]:
# Wrap back in the original structure
output_dict = {'teacher': averaged_state_dict}
print(f"Average dtype: {output_dict['teacher']['backbone.cls_token'].dtype}")

Average dtype: torch.float32


In [8]:
# Save averaged checkpoint
output_dir = Path("/data/ratna_retrain/eval/averaged_87500_to_137500") # RSG - change this for a new range
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "teacher_checkpoint.pth"
torch.save(output_dict, output_path)
print(f"Saved to: {output_path}")

Saved to: /data/ratna_retrain/eval/averaged_87500_to_137500/teacher_checkpoint.pth


The End

In [ ]:
# ckpt_path = Path("/data/ratna_retrain/eval/averaged_87500_to_137500/teacher_checkpoint.pth")
# state_dict = torch.load(ckpt_path, map_location="cpu")
# print(state_dict)

{'teacher': {'backbone.cls_token': tensor([[[-0.0144,  0.0024,  0.0050,  ...,  0.0013, -0.0127,  0.0015]]]), 'backbone.pos_embed': tensor([[[-1.4372e-02,  2.4660e-03,  5.0274e-03,  ...,  1.3211e-03,
          -1.2684e-02,  1.4708e-03],
         [ 2.7641e-02,  2.1676e-03, -2.5766e-02,  ..., -3.8454e-03,
          -1.9531e-02,  2.7800e-03],
         [ 1.9692e-03, -3.2710e-03,  2.9159e-03,  ..., -1.5046e-03,
           2.4929e-03, -1.7415e-04],
         ...,
         [-1.5668e-02,  5.8337e-03, -2.4156e-02,  ...,  3.5995e-04,
           6.5298e-03, -1.7924e-03],
         [-4.1579e-05,  4.1720e-03, -3.5862e-03,  ...,  4.2713e-03,
           1.8170e-03, -1.0814e-04],
         [-1.3359e-04, -1.0304e-03, -1.0116e-02,  ...,  4.7480e-04,
           5.2293e-03,  7.3907e-04]]]), 'backbone.register_tokens': tensor([[[ 5.7793e-03,  2.0194e-03,  8.6919e-03,  ...,  7.9479e-04,
          -2.2021e-02,  2.3724e-03],
         [-1.8279e-01,  1.9240e-03, -5.7171e-03,  ..., -1.6682e-05,
           3.2663e-02